# 🏷️ Promotion Typo Resilience & Intent Benchmark Notebook
**Chatbot YuedPao - Promotion Typo Resilience Evaluation**

สมุดโน้ตทดสอบประสิทธิภาพของระบบ Intent Classification เมื่อผู้ใช้พิมพ์คำถามหมวดโปรโมชันผิด (Typo Resilience):
1. **Tier 0 Domain Vocab Edit Distance:** ตรวจสอบการแก้ไขคำพิมพ์ผิด เช่น `ปะจำวัน` $\rightarrow$ `ประจำวัน`, `เดอนนี้` $\rightarrow$ `เดือนนี้`, `โคดสวนลด` $\rightarrow$ `โค้ดส่วนลด`
2. **Tier 1 Priority Rules (Promotion Fast Path):** ตรวจสอบการจำแนกคำถามเป็น `promotion_discount` ทันทีด้วยเวลา $< 5\text{ ms}$
3. **Promotion RRF Search Integration:** ตรวจสอบการดึงการ์ดโปรโมชันจาก SQLite `promotions` และ ChromaDB `yuedpao_promotions_e5`

In [ ]:
# ติดตั้งและโหลดโมดูลที่จำเป็น
import os
import sys
import time
import pandas as pd
from typing import List, Dict, Any

sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.abspath("../../"))
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

from app.services.intent_service import IntentService
from app.services.promotion_service import PromotionService

print("✅ โหลด IntentService และ PromotionService เรียบร้อย!")



## 🧪 Step 1: ชุดข้อมูลคำถามโปรโมชันที่มีคำพิมพ์ผิด (Promotion Typo Benchmark Dataset)

In [ ]:
PROMOTION_TYPO_BENCHMARK = [
    {"id": 1, "query": "ขอดีลปะจำวันหน่อยครับ", "expected_intent": "promotion_discount", "expected_sub_type": "daily_deal", "note": "พิมพ์ผิด: ปะจำวัน -> ประจำวัน"},
    {"id": 2, "query": "โปรวันนีมีไรบ้าง", "expected_intent": "promotion_discount", "expected_sub_type": "daily_deal", "note": "พิมพ์ผิด: โปรวันนี -> โปรวันนี้"},
    {"id": 3, "query": "ขอดูแฟลชเซลวันนี้หน่อย", "expected_intent": "promotion_discount", "expected_sub_type": "daily_deal", "note": "คำทับศัพท์: แฟลชเซล -> Flash Sale"},
    {"id": 4, "query": "มีโปรประจำว้นไหม", "expected_intent": "promotion_discount", "expected_sub_type": "daily_deal", "note": "พิมพ์ผิด: ประจำว้น -> ประจำวัน"},
    {"id": 5, "query": "ดีลปะจำเดือนมีตัวไหนบ้าง", "expected_intent": "promotion_discount", "expected_sub_type": "monthly_deal", "note": "พิมพ์ผิด: ปะจำเดือน -> ประจำเดือน"},
    {"id": 6, "query": "โปรโมชันเดอนนี้มีอะไรบ้าง", "expected_intent": "promotion_discount", "expected_sub_type": "monthly_deal", "note": "พิมพ์ผิด: เดอนนี้ -> เดือนนี้"},
    {"id": 7, "query": "ขอดูโปรประจำเดิอนหน่อย", "expected_intent": "promotion_discount", "expected_sub_type": "monthly_deal", "note": "พิมพ์ผิด: ประจำเดิอน -> ประจำเดือน"},
    {"id": 8, "query": "มีกางเกงยีนส์ลดราขาไหม", "expected_intent": "promotion_discount", "expected_sub_type": "general_promo", "note": "พิมพ์ผิด: ลดราขา -> ลดราคา"},
    {"id": 9, "query": "ขอโคดสวนลดหน่อยครับ", "expected_intent": "promotion_discount", "expected_sub_type": "general_promo", "note": "พิมพ์ผิด: โคดสวนลด -> โค้ดส่วนลด"},
    {"id": 10, "query": "เสื้อโปโลมีโปรลดราคาใหม", "expected_intent": "promotion_discount", "expected_sub_type": "general_promo", "note": "พิมพ์ผิด: ใหม -> ไหม"},
    {"id": 11, "query": "อยากได้เสื้อยืดงบไม่เกิน 300 มีโปรส่งฟรีใหม", "expected_intent": "promotion_discount", "expected_sub_type": "general_promo", "note": "คำสั่งประโยคยาวร่วมกับโปรส่งฟรี"},
    {"id": 12, "query": "มีคูปองสวนลดอะไรบ้างช่วงนี้", "expected_intent": "promotion_discount", "expected_sub_type": "general_promo", "note": "พิมพ์ผิด: สวนลด -> ส่วนลด"},
    {"id": 13, "query": "เสื้อโปโลลดราคาเหลือเท่าไหร่", "expected_intent": "promotion_discount", "expected_sub_type": "general_promo", "note": "คำถามสอบถามราคาส่วนลด"},
    {"id": 14, "query": "ขอโปรโมชันลดราคา 25%", "expected_intent": "promotion_discount", "expected_sub_type": "general_promo", "note": "ระบุตัวเลขเปอร์เซ็นต์ส่วนลด"},
    {"id": 15, "query": "มีดีลพิเศษอะไรน่าซื้อบ้างวันนี", "expected_intent": "promotion_discount", "expected_sub_type": "daily_deal", "note": "พิมพ์ผิด: วันนี -> วันนี้"}
]

print(f"🎉 โหลดชุดข้อมูลทดสอบคำถามโปรโมชันพิมพ์ผิดรวม {len(PROMOTION_TYPO_BENCHMARK)} รายการเรียบร้อย!")

## ⚡ Step 2: รันการทดสอบ Intent Classification & Spell Correction Benchmark

In [ ]:
intent_service = IntentService()
results = []
correct_count = 0

print("⏳ กำลังเริ่มทดสอบ Intent Engine บนประโยคโปรโมชันพิมพ์ผิด...\n")

for item in PROMOTION_TYPO_BENCHMARK:
    q_id = item["id"]
    raw_q = item["query"]
    expected = item["expected_intent"]
    
    # รัน Intent Prediction
    res = intent_service.predict_intent(raw_q)
    pred_intent = res["intent"]
    corrected_q = res.get("corrected_query", raw_q)
    tier_used = res.get("tier_used", "N/A")
    latency = res.get("latency_ms", 0.0)
    confidence = res.get("confidence", 0.0)
    
    is_correct = (pred_intent == expected)
    if is_correct:
        correct_count += 1
        status_str = "✅ ผ่าน"
    else:
        status_str = "❌ พลาด"
        
    results.append({
        "ID": q_id,
        "Raw Query (พิมพ์ผิด)": raw_q,
        "Corrected Query (คลีนแล้ว)": corrected_q,
        "Expected": expected,
        "Predicted": pred_intent,
        "Tier Used": tier_used,
        "Latency (ms)": f"{latency:.2f} ms",
        "Status": status_str,
        "Note": item["note"]
    })
    
    print(f"[{status_str}] [{q_id:02d}] '{raw_q}' -> คลีน: '{corrected_q}'")
    print(f"         ผลทำนาย: {pred_intent} ({tier_used} | {latency:.2f} ms)")

acc = (correct_count / len(PROMOTION_TYPO_BENCHMARK)) * 100.0
print(f"\n========================================================")
print(f"🏆 สรุปผลความแม่นยำ Intent Promotion Typo Accuracy: {acc:.2f}% ({correct_count}/{len(PROMOTION_TYPO_BENCHMARK)})")
print(f"========================================================")

## 🛍️ Step 3: ทดสอบการเชื่อมโยงดึงการ์ดสินค้าโปรโมชันจริงจาก PromotionService

In [ ]:
promo_service = PromotionService.get_instance()
sample_typo_queries = [
    "ขอดีลปะจำวันหน่อยครับ",
    "โปรโมชันเดอนนี้มีอะไรบ้าง",
    "มีกางเกงยีนส์ลดราขาไหม"
]

print("⏳ ทดลองยิงประโยคพิมพ์ผิดเข้า Promotion RRF Hybrid Search Engine...\n")
for sq in sample_typo_queries:
    search_res = promo_service.search_promotions(sq, top_k=3)
    print(f"💬 คำถามพิมพ์ผิด: '{sq}'")
    print(f"   🎉 ดึงสินค้าโปรโมชันได้ {len(search_res)} รายการ:")
    for p in search_res:
        print(f"     • {p['name']} | ราคาโปร: ฿{p['deal_price']} (ปกติ ฿{p['original_price']}) | ส่วนลด: {p['discount_tag']} | สี: {p['colors']}")
    print("-" * 70)

## 📊 Step 4: สรุปตารางรายงานผลลัพธ์ (Pandas DataFrame Summary)

In [ ]:
df_report = pd.DataFrame(results)
print(f"📊 รายงานผลการประเมิน Promotion Typo Resilience (Accuracy: {acc:.2f}%):")
print(df_report[["ID", "Raw Query (พิมพ์ผิด)", "Corrected Query (คลีนแล้ว)", "Predicted", "Tier Used", "Latency (ms)", "Status"]].to_string(index=False))